In [1]:
from google.colab import drive
import os

drive.mount('/content/drive')
PROJECT_ROOT = "/content/drive/MyDrive/Colab Notebooks/Porfolio/Balearia/2 Demanda"

os.makedirs(f"{PROJECT_ROOT}/data/processed", exist_ok=True)
os.makedirs(f"{PROJECT_ROOT}/reports", exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)

Mounted at /content/drive
PROJECT_ROOT: /content/drive/MyDrive/Colab Notebooks/Porfolio/Balearia/2 Demanda


In [2]:
import numpy as np
import pandas as pd

baseline = pd.read_parquet(f"{PROJECT_ROOT}/data/processed/baseline_predictions.parquet")
ml_test  = pd.read_parquet(f"{PROJECT_ROOT}/data/processed/ml_predictions.parquet")

baseline["departure_datetime_local"] = pd.to_datetime(baseline["departure_datetime_local"])
ml_test["departure_datetime_local"]  = pd.to_datetime(ml_test["departure_datetime_local"])

cutoff = pd.Timestamp("2025-07-01")
base_test = baseline[baseline["departure_datetime_local"] >= cutoff].copy()

print("base_test:", base_test.shape, "ml_test:", ml_test.shape)

base_test: (2346, 42) ml_test: (2346, 37)


In [3]:
def wape(y_true, y_pred):
    denom = np.sum(np.abs(y_true))
    return np.nan if denom == 0 else np.sum(np.abs(y_true - y_pred)) / denom

eval_df = base_test.merge(
    ml_test[["trip_id","pax_pred_ml"]],
    on="trip_id",
    how="inner"
)

y = eval_df["pax_real"].values
b = eval_df["pax_pred_baseline"].values
m = eval_df["pax_pred_ml"].values

metrics_global = pd.DataFrame([{
    "cutoff": str(cutoff.date()),
    "n_test": int(len(eval_df)),
    "MAE_baseline": float(np.mean(np.abs(y - b))),
    "WAPE_baseline": float(wape(y, b)),
    "MAE_ml": float(np.mean(np.abs(y - m))),
    "WAPE_ml": float(wape(y, m)),
    "WAPE_improvement_pp": float((wape(y, b) - wape(y, m)) * 100),  # puntos porcentuales
}])

display(metrics_global)

metrics_global.to_csv(f"{PROJECT_ROOT}/reports/compare_baseline_vs_ml_global.csv", index=False)
print("Saved ✅ compare_baseline_vs_ml_global.csv")

,cutoff,n_test,MAE_baseline,WAPE_baseline,MAE_ml,WAPE_ml,WAPE_improvement_pp
0,2025-07-01,2346,90.330672,0.129309,66.502691,0.095199,3.411003


Saved ✅ compare_baseline_vs_ml_global.csv


In [4]:
per_route = (
    eval_df.groupby("route_id")
    .apply(lambda g: pd.Series({
        "n": len(g),
        "WAPE_baseline": wape(g["pax_real"].values, g["pax_pred_baseline"].values),
        "WAPE_ml": wape(g["pax_real"].values, g["pax_pred_ml"].values),
        "MAE_baseline": np.mean(np.abs(g["pax_real"] - g["pax_pred_baseline"])),
        "MAE_ml": np.mean(np.abs(g["pax_real"] - g["pax_pred_ml"])),
        "WAPE_improvement_pp": (wape(g["pax_real"].values, g["pax_pred_baseline"].values)
                               - wape(g["pax_real"].values, g["pax_pred_ml"].values)) * 100
    }))
    .reset_index()
    .sort_values("WAPE_ml")
)

display(per_route)

per_route.to_csv(f"{PROJECT_ROOT}/reports/compare_baseline_vs_ml_per_route.csv", index=False)
print("Saved ✅ compare_baseline_vs_ml_per_route.csv")

/tmp/ipykernel_1267/1863588326.py:3: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: pd.Series({


,route_id,n,WAPE_baseline,WAPE_ml,MAE_baseline,MAE_ml,WAPE_improvement_pp
1,DEN-IBZ,665.0,0.110079,0.081817,87.582160,65.095987,2.826210
0,DEN-FOR,665.0,0.117720,0.093419,84.151381,66.779917,2.430112
3,VAL-IBZ,508.0,0.149772,0.105083,83.363893,58.489920,4.468863
2,DEN-PMI,508.0,0.157460,0.109796,108.984436,75.994012,4.766440


Saved ✅ compare_baseline_vs_ml_per_route.csv


In [5]:
tower = eval_df.copy()

# Ocupación forecast usando ML (puedes cambiar a baseline si quieres)
tower["occ_pred_ml"] = tower["pax_pred_ml"] / tower["capacity_pax"]

# Semáforo (RAG)
tower["flag"] = np.select(
    [
        tower["occ_pred_ml"] >= 0.92,
        tower["occ_pred_ml"].between(0.80, 0.919999),
        tower["occ_pred_ml"] < 0.60
    ],
    ["ROJO", "ÁMBAR", "VERDE"],
    default="NORMAL"
)

# Playbook (acciones sugeridas)
tower["action"] = np.select(
    [
        tower["flag"].eq("ROJO"),
        tower["flag"].eq("VERDE"),
        tower["flag"].eq("ÁMBAR"),
    ],
    [
        "Evaluar refuerzo/cambio buque + ajustar inventario/precio",
        "Activar promo táctica / bundles / campañas por ruta",
        "Monitorizar curva D-14/D-7 y microajustes precio/comunicación"
    ],
    default="Monitor estándar"
)

# Un par de columnas extra útiles para BI
tower["abs_error_baseline"] = np.abs(tower["pax_real"] - tower["pax_pred_baseline"])
tower["abs_error_ml"] = np.abs(tower["pax_real"] - tower["pax_pred_ml"])

tower.to_parquet(f"{PROJECT_ROOT}/data/processed/control_tower_table.parquet", index=False)
tower.to_csv(f"{PROJECT_ROOT}/data/processed/control_tower_table.csv", index=False)

print("Saved ✅ control_tower_table.parquet + control_tower_table.csv")
tower[["route_id","departure_datetime_local","pax_real","pax_pred_ml","occ_pred_ml","flag","action"]].head(15)

Saved ✅ control_tower_table.parquet + control_tower_table.csv


,route_id,departure_datetime_local,pax_real,pax_pred_ml,occ_pred_ml,flag,action
0,DEN-FOR,2025-07-01 08:00:00,690,692.788025,0.769764,NORMAL,Monitor estándar
1,DEN-FOR,2025-07-01 12:00:00,801,752.768250,0.836409,ÁMBAR,Monitorizar curva D-14/D-7 y microajustes prec...
2,DEN-FOR,2025-07-01 17:00:00,755,715.241577,0.596035,VERDE,Activar promo táctica / bundles / campañas por...
3,DEN-FOR,2025-07-01 20:00:00,900,692.716064,0.769685,NORMAL,Monitor estándar
4,DEN-FOR,2025-07-02 08:00:00,701,767.273621,0.852526,ÁMBAR,Monitorizar curva D-14/D-7 y microajustes prec...
5,DEN-FOR,2025-07-02 12:00:00,877,742.100281,0.618417,NORMAL,Monitor estándar
6,DEN-FOR,2025-07-02 17:00:00,858,714.288818,0.793654,NORMAL,Monitor estándar
7,DEN-FOR,2025-07-02 20:00:00,868,769.789673,0.855322,ÁMBAR,Monitorizar curva D-14/D-7 y microajustes prec...
8,DEN-FOR,2025-07-03 08:00:00,719,733.665955,0.611388,NORMAL,Monitor estándar
9,DEN-FOR,2025-07-03 12:00:00,840,716.349792,0.795944,NORMAL,Monitor estándar


In [6]:
flag_summary = tower["flag"].value_counts().to_frame("count")
flag_summary["share"] = flag_summary["count"] / flag_summary["count"].sum()

display(flag_summary)
flag_summary.to_csv(f"{PROJECT_ROOT}/reports/control_tower_flag_summary.csv")
print("Saved ✅ control_tower_flag_summary.csv")

,count,share
flag,,
VERDE,972,0.414322
NORMAL,793,0.338022
ÁMBAR,338,0.144075
ROJO,243,0.103581


Saved ✅ control_tower_flag_summary.csv
